[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_03/NB_bishop_ch03_distributions.ipynb)

# Chapter 3: Standard Distributions

This notebook explores key probability distributions from Bishop's *Deep Learning: Foundations and Concepts*, Chapter 3: discrete distributions (Bernoulli, Binomial, Multinomial), the multivariate Gaussian, kernel density estimation, and Gaussian mixtures.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats
from matplotlib.patches import Ellipse

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            legend._ncols = min(len(legend.get_texts()), 4)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)

## 1. Bernoulli Distribution

$\text{Bern}(x | \mu) = \mu^x (1-\mu)^{1-x}$, where $x \in \{0, 1\}$.

In [ ]:
mus = [0.2, 0.5, 0.8]
x_vals = [0, 1]

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, mu in zip(axes, mus):
    probs = [1 - mu, mu]
    ax.bar(x_vals, probs, color=['#1a3a6e', '#cd0000'], width=0.4)
    ax.set_xticks([0, 1])
    ax.set_ylim(0, 1)
    ax.set_xlabel('x')
    ax.set_ylabel('p(x)')
    ax.set_title(f'Bernoulli($\\mu$ = {mu})')

fig.suptitle('Bishop Ch 3: Bernoulli Distribution', fontsize=13, y=1.05)
fig.tight_layout()
plt.show()

## 2. Binomial Distribution

$\text{Bin}(m | N, \mu) = \binom{N}{m} \mu^m (1-\mu)^{N-m}$

In [ ]:
N_trials = 20
mus_binom = [0.2, 0.5, 0.8]
colors_binom = ['#1a3a6e', '#cd0000', '#228B22']

fig, ax = plt.subplots(figsize=(8, 5))
m_vals = np.arange(0, N_trials + 1)

for mu, c in zip(mus_binom, colors_binom):
    pmf = stats.binom.pmf(m_vals, N_trials, mu)
    ax.plot(m_vals, pmf, 'o-', color=c, linewidth=1.5, markersize=5,
            label=f'$\\mu$ = {mu}')

ax.set_xlabel('m (number of successes)')
ax.set_ylabel('p(m)')
ax.set_title(f'Binomial Distribution (N = {N_trials})')
ax.legend()
fig.tight_layout()
plt.show()

## 3. Multinomial Distribution: Dice Simulation

In [ ]:
np.random.seed(42)

# Fair die
probs_fair = np.ones(6) / 6
# Loaded die
probs_loaded = np.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.5])

n_rolls = 1000
rolls_fair = np.random.choice(6, size=n_rolls, p=probs_fair)
rolls_loaded = np.random.choice(6, size=n_rolls, p=probs_loaded)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

faces = np.arange(1, 7)
counts_fair = np.bincount(rolls_fair, minlength=6)
counts_loaded = np.bincount(rolls_loaded, minlength=6)

ax1.bar(faces, counts_fair / n_rolls, color='#1a3a6e', alpha=0.8)
ax1.axhline(1/6, color='#cd0000', linestyle='--', label='Expected (1/6)')
ax1.set_xlabel('Face')
ax1.set_ylabel('Relative Frequency')
ax1.set_title('Fair Die')
ax1.legend()

ax2.bar(faces, counts_loaded / n_rolls, color='#228B22', alpha=0.8)
ax2.bar(faces, probs_loaded, color='#cd0000', alpha=0.3, label='True probs')
ax2.set_xlabel('Face')
ax2.set_ylabel('Relative Frequency')
ax2.set_title('Loaded Die')
ax2.legend()

fig.suptitle('Bishop Ch 3: Multinomial Sampling', fontsize=13, y=1.03)
fig.tight_layout()
plt.show()

## 4. Discrete Distributions Summary Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Bernoulli
ax = axes[0]
mu = 0.3
ax.bar([0, 1], [1 - mu, mu], color=['#1a3a6e', '#cd0000'], width=0.4)
ax.set_title(f'Bernoulli($\\mu$={mu})')
ax.set_xticks([0, 1])
ax.set_ylim(0, 1)

# Binomial
ax = axes[1]
m = np.arange(0, 21)
ax.bar(m, stats.binom.pmf(m, 20, 0.3), color='#1a3a6e', alpha=0.8)
ax.set_title('Binomial(N=20, $\\mu$=0.3)')

# Poisson (bonus)
ax = axes[2]
k = np.arange(0, 20)
for lam, c in zip([2, 5, 10], ['#1a3a6e', '#cd0000', '#228B22']):
    ax.plot(k, stats.poisson.pmf(k, lam), 'o-', color=c, markersize=4,
            label=f'$\\lambda$={lam}')
ax.set_title('Poisson Distribution')
ax.legend(fontsize=8)

fig.suptitle('Bishop Ch 3: Discrete Distributions', fontsize=13, y=1.03)
fig.tight_layout()
save_fig(fig, 'bishop_ch3_distributions')
plt.show()

## 5. Multivariate Gaussian Distribution

$$\mathcal{N}(\mathbf{x} | \boldsymbol{\mu}, \boldsymbol{\Sigma}) = \frac{1}{(2\pi)^{D/2} |\boldsymbol{\Sigma}|^{1/2}} \exp\left(-\frac{1}{2}(\mathbf{x} - \boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu})\right)$$

In [ ]:
def plot_gaussian_contours(ax, mu, Sigma, color, label):
    """Plot contour ellipses for a 2D Gaussian."""
    x = np.linspace(mu[0] - 4, mu[0] + 4, 200)
    y = np.linspace(mu[1] - 4, mu[1] + 4, 200)
    X, Y = np.meshgrid(x, y)
    pos = np.dstack((X, Y))
    rv = stats.multivariate_normal(mu, Sigma)
    Z = rv.pdf(pos)
    ax.contour(X, Y, Z, levels=5, colors=color, linewidths=1.5)
    ax.plot(*mu, 'x', color=color, markersize=10, markeredgewidth=2)
    ax.set_aspect('equal')
    ax.set_title(label)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Spherical
plot_gaussian_contours(axes[0], [0, 0],
                        [[1, 0], [0, 1]], '#1a3a6e', 'Spherical')

# Diagonal
plot_gaussian_contours(axes[1], [0, 0],
                        [[2, 0], [0, 0.5]], '#cd0000', 'Diagonal')

# Full
plot_gaussian_contours(axes[2], [0, 0],
                        [[2, 1.2], [1.2, 1]], '#228B22', 'Full Covariance')

for ax in axes:
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

fig.suptitle('Bishop Ch 3: Multivariate Gaussian Contours', fontsize=13, y=1.03)
fig.tight_layout()
save_fig(fig, 'bishop_ch3_gaussian_contours')
plt.show()

## 6. Sampling from Multivariate Gaussians

In [ ]:
np.random.seed(42)

mu = np.array([1, -1])
Sigma = np.array([[2.0, 0.8],
                   [0.8, 1.0]])

samples = np.random.multivariate_normal(mu, Sigma, 500)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(samples[:, 0], samples[:, 1], alpha=0.4, s=15, color='#1a3a6e')

# Add contours
x = np.linspace(-3, 5, 200)
y = np.linspace(-4, 2, 200)
X, Y = np.meshgrid(x, y)
pos = np.dstack((X, Y))
Z = stats.multivariate_normal(mu, Sigma).pdf(pos)
ax.contour(X, Y, Z, levels=5, colors='#cd0000', linewidths=1.5)

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Samples from 2D Gaussian with Contours')
ax.set_aspect('equal')
fig.tight_layout()
plt.show()

## 7. Marginal and Conditional Distributions

In [ ]:
# Conditional p(x2 | x1 = 2)
x1_cond = 2.0
mu_cond = mu[1] + Sigma[1, 0] / Sigma[0, 0] * (x1_cond - mu[0])
sigma_cond = Sigma[1, 1] - Sigma[1, 0] ** 2 / Sigma[0, 0]

print(f'Conditional p(x2 | x1 = {x1_cond}):')
print(f'  Mean = {mu_cond:.4f}')
print(f'  Variance = {sigma_cond:.4f}')

fig, ax = plt.subplots(figsize=(6, 4))
x2_range = np.linspace(-4, 2, 300)
ax.plot(x2_range, stats.norm.pdf(x2_range, mu[1], np.sqrt(Sigma[1, 1])),
        color='#1a3a6e', linewidth=2, label=f'Marginal p($x_2$)')
ax.plot(x2_range, stats.norm.pdf(x2_range, mu_cond, np.sqrt(sigma_cond)),
        color='#cd0000', linewidth=2, label=f'Conditional p($x_2$ | $x_1$ = {x1_cond})')
ax.set_xlabel('$x_2$')
ax.set_ylabel('Density')
ax.set_title('Marginal vs Conditional')
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 8. Kernel Density Estimation (KDE)

$$p(\mathbf{x}) = \frac{1}{N} \sum_{n=1}^{N} \frac{1}{h^D} k\!\left(\frac{\mathbf{x} - \mathbf{x}_n}{h}\right)$$

In [ ]:
np.random.seed(42)

# Generate bimodal data
data_kde = np.concatenate([
    np.random.normal(-2, 0.8, 200),
    np.random.normal(2, 1.2, 300)
])

x_kde = np.linspace(-6, 7, 500)
bandwidths = [0.1, 0.3, 0.8, 2.0]
colors_kde = ['#1a3a6e', '#cd0000', '#228B22', '#FF8C00']

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(data_kde, bins=40, density=True, alpha=0.3, color='gray', label='Histogram')

for h, c in zip(bandwidths, colors_kde):
    kde = stats.gaussian_kde(data_kde, bw_method=h / data_kde.std())
    ax.plot(x_kde, kde(x_kde), linewidth=2, color=c, label=f'h = {h}')

ax.set_xlabel('x')
ax.set_ylabel('Density')
ax.set_title('Bishop Ch 3: Kernel Density Estimation')
ax.legend()
fig.tight_layout()
save_fig(fig, 'bishop_ch3_kde')
plt.show()

## 9. Gaussian Mixture Model (GMM)

In [ ]:
# Define a GMM with 3 components
weights = [0.3, 0.5, 0.2]
means = [[-2, 0], [1, 2], [3, -1]]
covs = [[[0.5, 0.1], [0.1, 0.3]],
        [[1.0, -0.3], [-0.3, 0.8]],
        [[0.3, 0], [0, 0.6]]]

# Sample from GMM
np.random.seed(42)
n_samples = 1000
component_labels = np.random.choice(3, size=n_samples, p=weights)
samples_gmm = np.zeros((n_samples, 2))

for i in range(n_samples):
    k = component_labels[i]
    samples_gmm[i] = np.random.multivariate_normal(means[k], covs[k])

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
colors_gmm = ['#1a3a6e', '#cd0000', '#228B22']

for k in range(3):
    mask = component_labels == k
    ax.scatter(samples_gmm[mask, 0], samples_gmm[mask, 1],
               alpha=0.4, s=15, color=colors_gmm[k], label=f'Component {k+1}')

# Contours for each component
x_grid = np.linspace(-5, 6, 200)
y_grid = np.linspace(-3, 5, 200)
X_g, Y_g = np.meshgrid(x_grid, y_grid)
pos_g = np.dstack((X_g, Y_g))

for k in range(3):
    rv = stats.multivariate_normal(means[k], covs[k])
    Z_g = rv.pdf(pos_g)
    ax.contour(X_g, Y_g, Z_g, levels=3, colors=colors_gmm[k],
               linewidths=1.5, alpha=0.8)

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Gaussian Mixture Model (3 Components)')
ax.legend()
fig.tight_layout()
plt.show()

## 10. Summary

**Key takeaways from Bishop Chapter 3:**

- Bernoulli, Binomial, and Multinomial are the building blocks for discrete data modeling.
- The multivariate Gaussian is parameterized by mean and covariance; covariance structure controls the shape.
- Conditional and marginal distributions of a Gaussian are also Gaussian.
- KDE provides nonparametric density estimation; bandwidth is the critical hyperparameter.
- Gaussian Mixture Models offer flexible density estimation via weighted sums of Gaussians.